In [1]:
using CSV, DataFrames, Statistics, Printf, StatsBase, Plots

include("functions.jl")

data_dir  = "output"
burnin    = 500_000
thin      = 10
chain_ids = 1:3
outfile   = joinpath(data_dir, "rhat_summary.csv")

filename_for_chain(c) = joinpath(data_dir, "samples_chain_$(c).csv")

function load_chain_df(path::String; burnin::Int, thin::Int)
    df = CSV.read(path, DataFrame)
    idx = (burnin + 1):thin:nrow(df)
    return df[idx, :]
end

dfs = DataFrame[]
for c in chain_ids
    fpath = filename_for_chain(c)
    push!(dfs, load_chain_df(fpath; burnin=burnin, thin=thin))
end

n_keep = minimum(nrow.(dfs))
dfs = [df[1:n_keep, :] for df in dfs]

params = String.(names(dfs[1]))
params = [p for p in params if eltype(dfs[1][!, p]) <: Real]

rhat_vals = Float64[]
rhat_names = String[]

for p in params
    mat = Array{Float64}(undef, length(chain_ids), n_keep)
    for (i, df) in enumerate(dfs)
        mat[i, :] = Float64.(df[!, p])
    end
    push!(rhat_names, p)
    push!(rhat_vals, rhat_gelman_rubin(mat))
end

outdf = DataFrame(param = rhat_names, Rhat = rhat_vals)
CSV.write(outfile, outdf)

println("Wrote Rhat summary to: $outfile")


Wrote Rhat summary to: output/rhat_summary.csv
